# AI & Data Analytics for Agribusiness
**Student:** Samarth Purekar  
**Project:** Agricultural Productivity & Climate Risk Analysis  
**Dataset:** `cleaned_crop_rainfall_data.csv`  
**Coverage:** 14 Indian States · 10 Crops · 2005-2015 · 2,953 Records  
**Goal:** Decision-support BI analysis — yield trends, climate-driven risk, actionable efficiency opportunities.

---
### Stages
1. Setup & Data Loading
2. Stage 1 — EDA
3. Stage 2 — Executive KPIs
4. Stage 3 — Climate Risk
5. Stage 4 — Efficiency Opportunities
6. Consolidated Findings

## 0. Setup — Imports & Configuration

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#8b949e',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.6,
})

CROP_COLORS = {
    'Arhar/Tur': '#f97316', 'Bajra': '#a3e635', 'Cotton(Lint)': '#f43f5e',
    'Groundnut': '#fb923c', 'Jowar': '#86efac', 'Maize': '#fbbf24',
    'Rice': '#38bdf8', 'Soyabean': '#c084fc', 'Sugarcane': '#4ade80', 'Wheat': '#facc15'
}
print('Libraries loaded successfully')

## 1. Data Loading & Validation

In [ ]:
DATA_PATH = r'd:/IBM_FInal_Project/cleaned_crop_rainfall_data.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'cleaned_crop_rainfall_data.csv'

df = pd.read_csv(DATA_PATH)
df[['Area','Production','Annual_Rainfall_mm','Yield']] = df[['Area','Production','Annual_Rainfall_mm','Yield']].astype(float)
df['Crop_Year'] = df['Crop_Year'].astype(int)

print(f'Shape       : {df.shape}')
print(f'Null values : {df.isnull().sum().sum()}')
print(f'Years       : {sorted(df.Crop_Year.unique())}')
print(f'States      : {df.State_Name.nunique()}  |  Crops: {df.Crop.nunique()}  |  Seasons: {df.Season.nunique()}')
df.head(3)

In [ ]:
print('=== Sanity Checks: min / max / mean ===')
print(df[['Area','Production','Annual_Rainfall_mm','Yield']].describe().round(2))
print()
print('NOTE: Yield max=85.95 is Sugarcane. Always analyse Yield per-crop.')

## 2. Stage 1 — Exploratory Data Analysis

### 2.1 Yield Distribution by Crop

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7), facecolor='#0d1117')
fig.suptitle('Yield Distribution by Crop (2005-2015)', color='#e6edf3', fontsize=13)
axes = axes.flatten()
for i, crop in enumerate(sorted(df['Crop'].unique())):
    sub = df[df['Crop'] == crop]['Yield']
    ax = axes[i]
    ax.set_facecolor('#161b22')
    ax.hist(sub, bins=20, color=CROP_COLORS[crop], alpha=0.85, edgecolor='none')
    ax.axvline(sub.mean(), color='white', lw=1.5, ls='--', label=f'Mean={sub.mean():.2f}')
    ax.set_title(crop, color='#e6edf3', fontsize=9)
    ax.legend(fontsize=7, facecolor='#1c2230', edgecolor='#30363d', labelcolor='#8b949e')
    ax.set_xlabel('Yield (t/ha)', fontsize=7)
    for s in ax.spines.values(): s.set_edgecolor('#30363d')
plt.tight_layout()
plt.savefig('yield_distribution.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

### 2.2 Average Yield Rankings

In [ ]:
grain_df = df[df['Crop'] != 'Sugarcane'].copy()

crop_yield  = df.groupby('Crop')['Yield'].mean().sort_values(ascending=False)
state_yield = grain_df.groupby('State_Name')['Yield'].mean().sort_values(ascending=False)
season_yield= df.groupby('Season')['Yield'].mean().sort_values(ascending=False)

print('Avg Yield by Crop (all):'); print(crop_yield.round(3))
print('\nAvg Yield by State (grain only):'); print(state_yield.round(3))
print('\nAvg Yield by Season:'); print(season_yield.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0d1117')
grain_crop  = grain_df.groupby('Crop')['Yield'].mean().sort_values()
state_y     = grain_df.groupby('State_Name')['Yield'].mean().sort_values()
gmean       = state_y.mean()

axes[0].set_facecolor('#161b22')
grain_crop.plot(kind='barh', ax=axes[0], color=[CROP_COLORS[c] for c in grain_crop.index])
axes[0].set_title('Avg Yield by Crop (grain)', color='#e6edf3')
axes[0].set_xlabel('Avg Yield (t/ha)')
for s in axes[0].spines.values(): s.set_edgecolor('#30363d')

axes[1].set_facecolor('#161b22')
state_y.plot(kind='barh', ax=axes[1], color=['#3fb950' if v >= gmean else '#f85149' for v in state_y])
axes[1].axvline(gmean, color='#d29922', ls='--', lw=1.5, label=f'Mean={gmean:.2f}')
axes[1].set_title('Avg Yield by State (grain)', color='#e6edf3')
axes[1].legend(facecolor='#1c2230', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=8)
for s in axes[1].spines.values(): s.set_edgecolor('#30363d')

plt.tight_layout()
plt.savefig('yield_rankings.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

### 2.3 Yield Trend Over Years (by Crop)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 12), facecolor='#0d1117')
fig.suptitle('Yield Trend 2005-2015 by Crop', color='#e6edf3', fontsize=13)
axes = axes.flatten()
grain_crops = [c for c in sorted(df['Crop'].unique()) if c != 'Sugarcane']
for i, crop in enumerate(grain_crops):
    sub = df[df['Crop'] == crop].groupby('Crop_Year')['Yield'].mean()
    ax  = axes[i]
    ax.set_facecolor('#161b22')
    ax.plot(sub.index, sub.values, color=CROP_COLORS[crop], lw=2, marker='o', ms=4)
    ax.fill_between(sub.index, sub.values, alpha=0.1, color=CROP_COLORS[crop])
    ax.axhline(sub.mean(), color='#30363d', lw=1, ls='--')
    ax.set_title(crop, color='#e6edf3', fontsize=9)
    ax.set_xlabel('Year', fontsize=7); ax.set_ylabel('t/ha', fontsize=7)
    ax.grid(True, alpha=0.3)
    for s in ax.spines.values(): s.set_edgecolor('#30363d')
axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('yield_trend_by_crop.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

### 2.4 Production & Yield Trend Over Years

In [ ]:
yr_grp  = df.groupby('Crop_Year').agg(Total_Prod=('Production','sum'), Avg_Rain=('Annual_Rainfall_mm','mean'))
grain_yr= df[df['Crop']!='Sugarcane'].groupby('Crop_Year')['Yield'].mean()

fig, ax1 = plt.subplots(figsize=(13, 5), facecolor='#0d1117')
ax1.set_facecolor('#161b22')
ax2 = ax1.twinx()

l1, = ax1.plot(yr_grp.index, yr_grp['Total_Prod']/1e6, color='#58a6ff', lw=2.5, marker='o', ms=5, label='Total Production (M t)')
l2, = ax2.plot(grain_yr.index, grain_yr.values,          color='#3fb950', lw=2, marker='s', ms=4, ls='--', label='Avg Yield grain (t/ha)')
l3, = ax2.plot(yr_grp.index, yr_grp['Avg_Rain']/100,     color='#d29922', lw=1.5, marker='^', ms=4, ls=':', label='Avg Rainfall (mm/100)')

ax1.set_xlabel('Year'); ax1.set_ylabel('Total Production (M t)', color='#58a6ff')
ax2.set_ylabel('Yield (t/ha) / Rainfall / 100', color='#3fb950')
ax1.set_title('Production & Yield Trend 2005-2015', color='#e6edf3', fontsize=12)
for s in ax1.spines.values(): s.set_edgecolor('#30363d')
ax1.grid(True, alpha=0.3)
plt.legend(handles=[l1,l2,l3], facecolor='#1c2230', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=9)
plt.tight_layout()
plt.savefig('production_yield_trend.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

### 2.5 Rainfall vs Yield Scatter (per Crop)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7), facecolor='#0d1117')
fig.suptitle('Annual Rainfall vs Yield — by Crop', color='#e6edf3', fontsize=13)
axes = axes.flatten()
for i, crop in enumerate(sorted(df['Crop'].unique())):
    sub = df[df['Crop'] == crop]
    ax  = axes[i]
    ax.set_facecolor('#161b22')
    ax.scatter(sub['Annual_Rainfall_mm'], sub['Yield'], color=CROP_COLORS[crop], alpha=0.5, s=15, edgecolors='none')
    m, b, r, _, _ = stats.linregress(sub['Annual_Rainfall_mm'], sub['Yield'])
    xr = np.linspace(sub['Annual_Rainfall_mm'].min(), sub['Annual_Rainfall_mm'].max(), 100)
    ax.plot(xr, m*xr+b, color='white', lw=1, alpha=0.7)
    ax.set_title(f'{crop}\nr={r:.3f}', color='#e6edf3', fontsize=8)
    ax.set_xlabel('Rainfall (mm)', fontsize=7); ax.set_ylabel('Yield (t/ha)', fontsize=7)
    for s in ax.spines.values(): s.set_edgecolor('#30363d')
plt.tight_layout()
plt.savefig('rainfall_yield_scatter.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Note: Aggregate r values are weak (0.04-0.12). Year-over-year change analysis reveals stronger climate signals.')

## 3. Stage 2 — Executive KPIs

In [ ]:
grain_df = df[df['Crop'] != 'Sugarcane']

kpi_total_prod = df['Production'].sum()
kpi_total_area = df['Area'].sum()
kpi_avg_yield  = grain_df['Yield'].mean()
kpi_best_state = grain_df.groupby('State_Name')['Yield'].mean().idxmax()
kpi_best_v     = grain_df.groupby('State_Name')['Yield'].mean().max()
kpi_worst_st   = grain_df.groupby('State_Name')['Yield'].mean().idxmin()
kpi_worst_v    = grain_df.groupby('State_Name')['Yield'].mean().min()
kpi_avg_rain   = df['Annual_Rainfall_mm'].mean()

wheat_yr = df[df['Crop']=='Wheat'].groupby('Crop_Year')['Yield'].mean()
kpi_yoy  = (wheat_yr.iloc[-1]-wheat_yr.iloc[-2])/wheat_yr.iloc[-2]*100

ups = df[(df['State_Name']=='Uttar Pradesh') & (df['Crop']=='Sugarcane')].groupby('Crop_Year').agg({'Annual_Rainfall_mm':'mean','Yield':'mean'})
kpi_rf_r, _ = stats.pearsonr(ups['Annual_Rainfall_mm'].diff().dropna(), ups['Yield'].diff().dropna())

crop_mean_yield = df.groupby('Crop')['Yield'].mean()
crop_mean_area  = df.groupby('Crop')['Area'].mean()
gaps = []
for (st, cr), grp in df.groupby(['State_Name','Crop']):
    if grp['Area'].mean() > crop_mean_area[cr] and grp['Yield'].mean() < crop_mean_yield[cr]:
        gaps.append({'State':st,'Crop':cr,
                     'AvgArea':grp['Area'].mean(),'AvgYield':grp['Yield'].mean(),
                     'CropMeanYield':crop_mean_yield[cr],
                     'GapPct':(crop_mean_yield[cr]-grp['Yield'].mean())/crop_mean_yield[cr]*100,
                     'AvgRainfall':grp['Annual_Rainfall_mm'].mean()})
gaps_df = pd.DataFrame(gaps).sort_values('GapPct', ascending=False)

print('='*62)
print(f'KPI 1 | Total Production           : {kpi_total_prod:>12,.0f} tonnes')
print(f'KPI 2 | Cultivated Area            : {kpi_total_area:>12,.0f} hectares')
print(f'KPI 3 | Avg Yield (grain)          : {kpi_avg_yield:>12.3f} t/ha')
print(f'KPI 4 | YoY Yield Growth (Wheat)   : {kpi_yoy:>+11.2f}%')
print(f'KPI 5 | Avg Annual Rainfall        : {kpi_avg_rain:>12.1f} mm')
print(f'KPI 6 | RF-Yield Corr (UP Sugar)   : {kpi_rf_r:>12.3f}')
print(f'KPI 7 | Efficiency Gap Opptys      : {len(gaps_df):>12} state-crop cells')
print(f'       | Most Efficient State      : {kpi_best_state} ({kpi_best_v:.2f} t/ha)')
print(f'       | Least Efficient State     : {kpi_worst_st} ({kpi_worst_v:.2f} t/ha)')
print('='*62)

### 3.1 State x Crop Heatmap

In [ ]:
hm      = df.groupby(['State_Name','Crop'])['Yield'].mean().unstack(fill_value=0)
hm_norm = (hm - hm.min()) / (hm.max() - hm.min())

fig, ax = plt.subplots(figsize=(14, 8), facecolor='#0d1117')
ax.set_facecolor('#161b22')
cmap = sns.diverging_palette(10, 130, l=40, as_cmap=True)
sns.heatmap(hm_norm, ax=ax, cmap=cmap, annot=hm.round(2), fmt='.2f',
            annot_kws={'size':7}, linewidths=0.4, linecolor='#0d1117',
            cbar_kws={'label':'Normalised Yield (per crop)'})
ax.set_title('State x Crop Average Yield Heatmap (per-column normalised)', color='#e6edf3', fontsize=11)
ax.set_xlabel('Crop', color='#8b949e'); ax.set_ylabel('State', color='#8b949e')
plt.xticks(rotation=30, ha='right', color='#8b949e', fontsize=8)
plt.yticks(rotation=0, color='#8b949e', fontsize=8)
ax.collections[0].colorbar.ax.yaxis.label.set_color('#8b949e')
ax.collections[0].colorbar.ax.tick_params(colors='#8b949e')
plt.tight_layout()
plt.savefig('heatmap_state_crop.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 4. Stage 3 — Climate Risk Analysis

### 4.1 Yield Volatility — Coefficient of Variation (CV)

In [ ]:
cv_rows = []
for (st, cr), grp in df[df['Crop']!='Sugarcane'].groupby(['State_Name','Crop']):
    yr_means = grp.groupby('Crop_Year')['Yield'].mean()
    if len(yr_means) >= 4:
        c_val = yr_means.std() / yr_means.mean() if yr_means.mean() > 0 else 0
        cv_rows.append({'State':st,'Crop':cr,'CV':c_val,'AvgYield':yr_means.mean(),'Years':len(yr_means)})
cv_df = pd.DataFrame(cv_rows).sort_values('CV', ascending=False)
print('Top 20 High-Volatility State x Crop Pairs:')
print(cv_df.head(20)[['State','Crop','CV','AvgYield']].round(3).to_string(index=False))

In [ ]:
top_cv = cv_df.head(15).copy()
top_cv['Label'] = top_cv['State'].str[:8] + ' . ' + top_cv['Crop'].str[:8]
bc = ['#f85149' if v>0.22 else '#d29922' if v>0.18 else '#3fb950' for v in top_cv['CV']]

fig, ax = plt.subplots(figsize=(12, 6), facecolor='#0d1117')
ax.set_facecolor('#161b22')
ax.barh(top_cv['Label'][::-1], top_cv['CV'][::-1], color=bc[::-1], height=0.7)
ax.axvline(0.20, color='#d29922', lw=1.5, ls='--', label='High Risk (CV=0.20)')
ax.set_title('Top 15 State x Crop Yield Volatility (CV)', color='#e6edf3')
ax.set_xlabel('Coefficient of Variation (CV)')
ax.legend(facecolor='#1c2230', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=8)
for s in ax.spines.values(): s.set_edgecolor('#30363d')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('yield_volatility.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

### 4.2 Rainfall Change vs Yield Change Correlation

In [ ]:
rf_rows = []
for (st, cr), grp in df.groupby(['State_Name','Crop']):
    yr_agg = grp.groupby('Crop_Year').agg({'Annual_Rainfall_mm':'mean','Yield':'mean'}).sort_index()
    if len(yr_agg) < 5:
        continue
    rf_chg = yr_agg['Annual_Rainfall_mm'].diff().dropna()
    y_chg  = yr_agg['Yield'].diff().dropna()
    if len(rf_chg) >= 4:
        r_val, p_val = stats.pearsonr(rf_chg, y_chg)
        rf_rows.append({'State':st,'Crop':cr,'RF_Yield_Corr':r_val,'p_value':p_val})
rf_df = pd.DataFrame(rf_rows)
rf_df['AbsCorr'] = rf_df['RF_Yield_Corr'].abs()
rf_df = rf_df.sort_values('AbsCorr', ascending=False)
print('Top 20 DeltaRF -> DeltaYield Correlations:')
print(rf_df.head(20)[['State','Crop','RF_Yield_Corr','p_value']].round(3).to_string(index=False))
print('WARNING: Correlation != causation. Confounders: irrigation, inputs, policy.')

In [ ]:
top_rf = rf_df.head(12).copy()
top_rf['Label'] = top_rf['State'].str[:10] + ' . ' + top_rf['Crop'].str[:8]
bc2 = ['#f85149' if v>0.85 else '#d29922' if v>0.75 else '#3fb950' for v in top_rf['RF_Yield_Corr']]

fig, ax = plt.subplots(figsize=(12, 5), facecolor='#0d1117')
ax.set_facecolor('#161b22')
ax.barh(top_rf['Label'][::-1], top_rf['RF_Yield_Corr'][::-1], color=bc2[::-1], height=0.7)
ax.axvline(0.80, color='#f85149', lw=1.2, ls='--', label='Critical (r=0.80)')
ax.set_title('Delta-Rainfall to Delta-Yield Correlation (Top Climate-Risk Pairs)', color='#e6edf3')
ax.set_xlabel('Pearson r (year-over-year changes)')
ax.legend(facecolor='#1c2230', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=8)
for s in ax.spines.values(): s.set_edgecolor('#30363d')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('rf_yield_corr.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5. Stage 4 — Efficiency Gap Opportunities

In [ ]:
print('Efficiency Gap Opportunities (High Area + Below-avg Yield):')
print(gaps_df.head(15)[['State','Crop','AvgArea','AvgYield','CropMeanYield','GapPct','AvgRainfall']].round(2).to_string(index=False))

In [ ]:
top_g  = gaps_df.head(12).copy()
top_g['Label'] = top_g['State'].str[:10] + ' . ' + top_g['Crop'].str[:8]
cg = [CROP_COLORS.get(c, '#58a6ff') for c in top_g['Crop']]

fig, ax = plt.subplots(figsize=(12, 5), facecolor='#0d1117')
ax.set_facecolor('#161b22')
ax.barh(top_g['Label'][::-1], top_g['GapPct'][::-1], color=cg[::-1], height=0.7)
ax.set_title('Efficiency Gap % (High Area, Below-Average Yield)', color='#e6edf3')
ax.set_xlabel('Yield Gap vs Crop Mean (%)')
for s in ax.spines.values(): s.set_edgecolor('#30363d')
ax.grid(True, axis='x', alpha=0.3)
for i, (_, row) in enumerate(top_g[::-1].iterrows()):
    ax.text(row['GapPct']+0.05, i, f"{row['GapPct']:.1f}%", va='center', fontsize=8, color='#e6edf3')
plt.tight_layout()
plt.savefig('efficiency_gaps.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 6. Consolidated Findings

In [ ]:
print('='*72)
print('  AGRIBUSINESS BI ANALYSIS - CONSOLIDATED FINDINGS')
print('='*72)
print()
print(f'DATASET: {len(df):,} records | {df.State_Name.nunique()} states | {df.Crop.nunique()} crops | 2005-2015')
print(f'Nulls  : {df.isnull().sum().sum()} (clean)')
print()
print('EXECUTIVE KPIs')
print(f'  Total Production      : {df["Production"].sum()/1e6:.2f} M tonnes')
print(f'  Avg Yield (grain)     : {grain_df["Yield"].mean():.3f} t/ha')
print(f'  Best State (grain)    : {kpi_best_state} ({kpi_best_v:.2f} t/ha)')
print(f'  Worst State (grain)   : {kpi_worst_st} ({kpi_worst_v:.2f} t/ha)')
print(f'  State gap             : {(kpi_best_v-kpi_worst_v)/kpi_best_v*100:.1f}%')
print()
print('TOP 5 CLIMATE RISKS')
for _, row in rf_df.head(5).iterrows():
    print(f'  {row["State"]:<22} {row["Crop"]:<20} r={row["RF_Yield_Corr"]:.3f}')
print()
print('TOP 5 YIELD VOLATILITY (CV)')
for _, row in cv_df.head(5).iterrows():
    print(f'  {row["State"]:<22} {row["Crop"]:<20} CV={row["CV"]:.3f}')
print()
print('TOP 5 EFFICIENCY GAPS')
for _, row in gaps_df.head(5).iterrows():
    print(f'  {row["State"]:<22} {row["Crop"]:<20} Gap={row["GapPct"]:.1f}%  Area={row["AvgArea"]:.0f}ha')
print()
print('  Dashboard: agri_dashboard.html (IBM Cognos-style, fully interactive)')
print('='*72)

---
**End of Analysis**  
*Samarth Purekar | AI & Data Analytics for Agribusiness | IBM Project*